# Repaint a block copolymer chain

#### Length 80, Goal is 1:2:1 = 10:60:10

In [45]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 0. Sort by atom ID in OVITO, export sorted data file

## 1. Find atom section in data file

In [46]:
# find the line numbers of key sections
with open("sorted_tetrablock_data", 'r') as f:
    lines = f.readlines()

atoms_start = next(i for i, l in enumerate(lines) if l.strip() == 'Atoms  # molecular') + 1
velocities_start = next(i for i, l in enumerate(lines) if l.strip() == 'Velocities')

print(f"Atoms section starts at line: {atoms_start}")
print(f"Velocities section starts at line: {velocities_start}")

Atoms section starts at line: 21
Velocities section starts at line: 502903


## 2. Load atom section of datafile

In [47]:
headers = ['atom_id','mol_id','atom_type','x','y','z','ix','iy','iz']
df = pd.read_csv("sorted_tetrablock_data", names=headers, 
                 skiprows=atoms_start, 
                 nrows=velocities_start - atoms_start-2,  # -1 for the blank line
                 delimiter=' ')
print(len(df))
df

502880


,atom_id,mol_id,atom_type,x,y,z,ix,iy,iz
0,1,1,2,-26.688946,-43.735604,-7.454141,0,0,0
1,2,1,2,-26.919139,-42.915477,-7.123054,0,0,0
2,3,1,2,-27.023133,-42.772666,-6.156884,0,0,0
3,4,1,2,-27.297848,-43.116130,-5.146881,0,0,0
4,5,1,2,-26.384834,-43.137022,-4.702749,0,0,0
...,...,...,...,...,...,...,...,...,...
502875,502876,6286,3,37.576487,70.557047,36.798576,0,0,0
502876,502877,6286,3,37.770137,70.484896,35.825556,0,0,0
502877,502878,6286,3,37.448169,69.546257,36.272191,0,0,0
502878,502879,6286,3,37.836446,69.615694,35.346278,0,0,0


## 3. Calculate atom ID on a chain (1-80) and assign new types

In [48]:
df['id_by_chain']=((df['atom_id']-1) % 80)+1
id_by_chain = df['id_by_chain']
mask_type1 = (10 < id_by_chain) & (id_by_chain<= 70)
mask_type2 = ((1 <= id_by_chain) & (id_by_chain <= 10) | (70 < id_by_chain) & (id_by_chain <= 80))

In [49]:
type_list = []
type_list = np.select([mask_type1, mask_type2], [1, 2], default=0)
df['new_type'] = type_list

pd.set_option('display.max_rows', 200)
print('Print first 2 chains:')
df.head(160)

Print first 2 chains:


,atom_id,mol_id,atom_type,x,y,z,ix,iy,iz,id_by_chain,new_type
0,1,1,2,-26.688946,-43.735604,-7.454141,0,0,0,1,2
1,2,1,2,-26.919139,-42.915477,-7.123054,0,0,0,2,2
2,3,1,2,-27.023133,-42.772666,-6.156884,0,0,0,3,2
3,4,1,2,-27.297848,-43.116130,-5.146881,0,0,0,4,2
4,5,1,2,-26.384834,-43.137022,-4.702749,0,0,0,5,2
5,6,1,2,-26.004341,-43.134393,-3.760781,0,0,0,6,2
6,7,1,2,-25.388809,-43.906126,-3.191778,0,0,0,7,2
7,8,1,2,-24.502743,-43.603821,-2.699913,0,0,0,8,2
8,9,1,2,-24.313830,-44.426328,-2.288319,0,0,0,9,2
9,10,1,2,-23.672540,-43.699649,-2.103704,0,0,0,10,2


## 4. Replace old type with new type, add header and tail, and write to new file

In [50]:
df['atom_type'] = df['new_type']

# read the original header and everything from velocity
with open("sorted_tetrablock_data", 'r') as f:
    lines = f.readlines()

header_lines = lines[:atoms_start]
tail = lines[velocities_start-1:]

# write header + modified atoms + tail
with open("repainted_sorted_tetrablock_data", 'w') as f:
    f.writelines(header_lines)
    df[['atom_id','mol_id','atom_type','x','y','z','ix','iy','iz']].to_csv(
        f, sep=' ', index=False, header=False
    )
    f.writelines(tail)

print(repr(lines[velocities_start]))    # should be 'Velocities\n'
print(repr(lines[velocities_start-1]))  # should be '\n' (blank line)

'Velocities\n'
'\n'
